# LLM Fine-Tuning: JD Structured Extractor

QLoRA fine-tuning of Qwen2-0.5B on LinkedIn + hand-labeled JDs.

**Runtime: T4 GPU required**

In [ ]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate tiktoken kagglehub pandas
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import json, random, os
import kagglehub
import pandas as pd

print("[1/6] Downloading LinkedIn dataset from Kaggle...")
path = kagglehub.dataset_download("asaniczka/1-3m-linkedin-jobs-and-skills-2024")
print(f"Downloaded to: {path}")
print(os.listdir(path))

In [ ]:
print("[2/6] Processing dataset...")
jobs_df = pd.read_csv(os.path.join(path, "linkedin_job_postings.csv"))
skills_df = pd.read_csv(os.path.join(path, "job_skills.csv"))

ai_keywords = ['ai', 'machine learning', 'data scientist', 'nlp', 'deep learning',
               'ml engineer', 'artificial intelligence', 'computer vision', 'llm']
mask = jobs_df['job_title'].str.lower().str.contains('|'.join(ai_keywords), na=False)
ai_jobs = jobs_df[mask].head(300)
print(f"Found {len(ai_jobs)} AI/ML jobs from {len(jobs_df)} total")

merged = ai_jobs.merge(skills_df, on='job_link', how='left')
skills_by_job = merged.groupby('job_link')['job_skills'].apply(list).to_dict()

INSTRUCTION = "Extract structured information from the following job description. Return valid JSON with these fields: title, company, location, work_model, seniority, required_skills (list), nice_to_have (list), salary, language."

training_data = []
for _, row in ai_jobs.iterrows():
    job_skills = skills_by_job.get(row.get('job_link', ''), [])
    output = {
        "title": str(row.get("job_title", "")),
        "company": str(row.get("company", "")),
        "location": str(row.get("job_location", "")),
        "work_model": str(row.get("job_type", "")),
        "seniority": str(row.get("job_level", "")),
        "required_skills": [s for s in job_skills if pd.notna(s)][:10],
        "nice_to_have": [],
        "salary": str(row.get("salary", "")) if pd.notna(row.get("salary")) else "",
        "language": "English"
    }
    desc = str(row.get("job_description", ""))
    if len(desc) > 100:
        training_data.append({"instruction": INSTRUCTION, "input": desc[:2000], "output": json.dumps(output)})

random.seed(42)
random.shuffle(training_data)
train_n = int(len(training_data) * 0.85)
train_data = training_data[:train_n]
test_data = training_data[train_n:]

os.makedirs('data', exist_ok=True)
with open('data/train.jsonl', 'w') as f:
    for item in train_data:
        f.write(json.dumps(item) + '\n')
with open('data/test.jsonl', 'w') as f:
    for item in test_data:
        f.write(json.dumps(item) + '\n')

print(f"Done: {len(train_data)} train, {len(test_data)} test examples")

## Train with QLoRA

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

BASE_MODEL = "Qwen/Qwen2-0.5B-Instruct"
OUTPUT_DIR = "output/jd-extractor-qwen-0.5b-linkedin"

print(f"[3/6] Loading {BASE_MODEL}...")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model = prepare_model_for_kbit_training(model)

print("[4/6] Applying LoRA...")
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

dataset = load_dataset("json", data_files="data/train.jsonl", split="train")
print(f"{len(dataset)} training examples")

def format_prompt(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"}

dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)

print("[5/6] Training...")
training_args = SFTConfig(output_dir=OUTPUT_DIR, num_train_epochs=3, per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=2e-4, fp16=True, logging_steps=5, save_strategy="epoch", warmup_steps=10, lr_scheduler_type="cosine", report_to="none", optim="adamw_torch", max_length=1024)
trainer = SFTTrainer(model=model, train_dataset=dataset, args=training_args, processing_class=tokenizer)
trainer.train()

print("Saving adapter...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete!")

## Evaluate on Test Set

In [ ]:
from peft import PeftModel

print("[6/6] Evaluating...")
del model
del trainer
torch.cuda.empty_cache()

eval_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
eval_model = PeftModel.from_pretrained(eval_model, OUTPUT_DIR)
eval_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

test_data = []
with open('data/test.jsonl') as f:
    for line in f:
        test_data.append(json.loads(line))

STRING_FIELDS = ["title", "company", "location", "work_model", "seniority", "salary", "language"]
LIST_FIELDS = ["required_skills", "nice_to_have"]
results = {"json_valid": 0, "json_invalid": 0, "field_scores": []}

for i, ex in enumerate(test_data):
    prompt = f"### Instruction:\nExtract structured information from the following job description. Return valid JSON.\n\n### Input:\n{ex['input'][:1500]}\n\n### Response:\n"
    inputs = eval_tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    resp = eval_tokenizer.decode(out[0], skip_special_tokens=True)
    if "### Response:" in resp:
        resp = resp.split("### Response:")[-1].strip()
    try:
        parsed = json.loads(resp) if resp.startswith('{') else json.loads(resp[resp.find('{'):])
        results["json_valid"] += 1
        expected = json.loads(ex['output'])
        scores = {}
        for f in STRING_FIELDS:
            p, e = str(parsed.get(f,'')).lower().strip(), str(expected.get(f,'')).lower().strip()
            scores[f] = 1.0 if p == e else (0.5 if e in p or p in e else 0.0)
        for f in LIST_FIELDS:
            pl = {s.lower().strip() for s in parsed.get(f,[])}
            el = {s.lower().strip() for s in expected.get(f,[])}
            m = pl & el
            pr = len(m)/len(pl) if pl else 0
            rc = len(m)/len(el) if el else 0
            scores[f] = {"f1": 2*pr*rc/(pr+rc) if pr+rc else 0}
        results["field_scores"].append(scores)
    except:
        results["json_invalid"] += 1
    print(f"  [{i+1}/{len(test_data)}] valid: {results['json_valid']}/{results['json_valid']+results['json_invalid']}")

n = len(results["field_scores"])
if n:
    total_examples = results["json_valid"] + results["json_invalid"]
    print(f"\nJSON validity: {results['json_valid']}/{total_examples}")
    print("\nField accuracy:")
    for f in STRING_FIELDS:
        avg = sum(s[f] for s in results["field_scores"])/n
        print(f"  {f}: {avg:.2f}")
    for f in LIST_FIELDS:
        avg_f1 = sum(s[f]["f1"] for s in results["field_scores"])/n
        print(f"  {f}: F1={avg_f1:.2f}")

with open('eval_results.json', 'w') as out_f:
    json.dump(results, out_f, indent=2, default=str)
print("\nResults saved to eval_results.json")